[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/04_%E7%B5%B1%E8%A8%88_2%E7%B4%9A/08_%E5%AE%9F%E8%B7%B5_%E3%82%BF%E3%82%A4%E3%82%BF%E3%83%8B%E3%83%83%E3%82%AF%E5%88%86%E6%9E%90.ipynb)

# 統計2級-08. 実践：タイタニックを「検定」で分析する

> Colabで実行可。最初に「① セットアップ」セルを実行（Colabはデータを自動生成、ローカルは何もしない）。

[3級の実践ノート](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/03_%E7%B5%B1%E8%A8%88_3%E7%B4%9A/07_%E5%AE%9F%E8%B7%B5_%E3%82%BF%E3%82%A4%E3%82%BF%E3%83%8B%E3%83%83%E3%82%AF%E5%88%86%E6%9E%90.ipynb)では「女性の生存率が高い**ように見える**（0.74 vs 0.19）」まで読み取りました。
**この2級ノートでは、その差が“偶然ではない”かを検定で確かめます。**

使う2級の道具：**カイ二乗検定・母比率の差の検定・2標本t検定・区間推定・分散分析(ANOVA)・重回帰**。

In [ ]:
# === ① セットアップ（最初に実行してください）===
import pandas as pd               # 表データ
import numpy as np                # 数値計算
import matplotlib.pyplot as plt   # グラフ描画
from scipy import stats           # 検定・分布
import os
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の文字化け防止
# ローカルにあればそれを、無ければ公開URL（Kaggle trainと同じ内容）から読み込む
local = '../data/titanic.csv'
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(local) if os.path.exists(local) else pd.read_csv(url)  # データを読み込む
df['家族人数'] = df['SibSp'] + df['Parch'] + 1   # 本人＋同乗家族＝家族人数（後で使う）
print('乗客数:', len(df))
df.head()

## 1. 出発点：3級で見えた“差”

性別ごとの生存率をもう一度確認します。これが「本物の差」かを以降で検定します。

In [ ]:
print('性別ごとの生存率 P(生存 | 性別):')
print(df.groupby('Sex')['Survived'].mean().round(3))
print('全体:', round(df['Survived'].mean(), 3))

## 2. カイ二乗 独立性の検定（性別と生存は関連がある？）

- 帰無仮説 H₀：性別と生存は**無関係**（差は偶然）
- 対立仮説 H₁：性別と生存に**関連がある**

クロス集計表からカイ二乗検定をします。

In [ ]:
ct = pd.crosstab(df['Sex'], df['Survived'])         # 性別×生存 の人数表
print(ct)
chi2, p, dof, expected = stats.chi2_contingency(ct)  # カイ二乗 独立性の検定
print(f'\nカイ二乗 = {chi2:.1f}, 自由度 = {dof}, p値 = {p:.2e}')
print('結論:', '性別と生存に関連あり（有意）' if p < 0.05 else '関連があるとは言えない')

p値はほぼ0（≈10⁻⁵⁸）。**「偶然こんな差が出る確率はほぼゼロ」→ 性別と生存は確かに関連**しています。
3級の「ありそう」が、2級の検定で「確かにある」になりました。

## 3. 母比率の差の検定（女性と男性の生存率の差）

2グループの**割合の差**を直接検定します（A/Bテストと同じ手法）。

In [ ]:
from statsmodels.stats.proportion import proportions_ztest
f = df[df['Sex'] == 'female']['Survived']    # 女性の生存(0/1)
m = df[df['Sex'] == 'male']['Survived']      # 男性の生存(0/1)
z, p = proportions_ztest([f.sum(), m.sum()], [len(f), len(m)])  # 2群の比率の差の検定
print(f'女性 {f.mean():.3f} vs 男性 {m.mean():.3f}')
print(f'z = {z:.1f}, p値 = {p:.2e}')
print('結論:', '生存率に差あり（有意）' if p < 0.05 else '差は有意でない')

## 4. 2標本t検定（生存者と死亡者で運賃の平均は違う？）

今度は**量的データ（運賃）の平均の差**。等分散を仮定しないWelchのt検定を使います。

In [ ]:
s = df[df['Survived'] == 1]['Fare']    # 生存者の運賃
d = df[df['Survived'] == 0]['Fare']    # 死亡者の運賃
t, p = stats.ttest_ind(s, d, equal_var=False)   # Welchの2標本t検定
print(f'平均運賃  生存 {s.mean():.1f} / 死亡 {d.mean():.1f}')
print(f't = {t:.1f}, p値 = {p:.2e}')
print('結論:', '運賃の平均に差あり（生存者が高い）' if p < 0.05 else '差は有意でない')

生存者のほうが平均運賃が高い（48.4 vs 22.1）＝**高い運賃＝上等客ほど助かりやすかった**ことの裏づけ。

## 5. 区間推定（信頼区間）

1つの数（点推定）ではなく「だいたいこの範囲」と幅で示します。

In [ ]:
# (1) 全体の生存率の95%信頼区間（母比率の区間推定）
n = len(df); ph = df['Survived'].mean()
se = np.sqrt(ph * (1 - ph) / n)               # 比率の標準誤差
print(f'生存率 {ph:.3f}  95%CI [{ph - 1.96*se:.3f}, {ph + 1.96*se:.3f}]')

# (2) 平均運賃の95%信頼区間（母平均の区間推定・t分布）
fare = df['Fare']
ci = stats.t.interval(0.95, len(fare) - 1, loc=fare.mean(),
                      scale=fare.std(ddof=1) / np.sqrt(len(fare)))
print('平均運賃 95%CI:', np.round(ci, 1))

## 6. 一元配置分散分析（ANOVA：等級で運賃の平均は違う？）

3グループ以上（1等・2等・3等）の平均を**一度に**比較します。

In [ ]:
groups = [df[df['Pclass'] == c]['Fare'] for c in [1, 2, 3]]  # 等級ごとの運賃
F, p = stats.f_oneway(*groups)                  # 一元配置分散分析
print(f'F = {F:.1f}, p値 = {p:.2e}')
print('結論:', '等級によって平均運賃が違う（有意）' if p < 0.05 else '差は有意でない')
df.boxplot(column='Fare', by='Pclass'); plt.suptitle(''); plt.title('等級別の運賃'); plt.show()

## 7. 重回帰分析（運賃を複数の要因で説明する）

運賃 `Fare` を **等級・家族人数・年齢** で説明します（等級はダミー変数）。
偏回帰係数・決定係数・各係数のp値を読みます。

In [ ]:
import statsmodels.formula.api as smf
d2 = df.dropna(subset=['Age'])                 # Ageの欠損行を除く
model = smf.ols('Fare ~ C(Pclass) + 家族人数 + Age', data=d2).fit()  # 重回帰
print('決定係数 R^2:', round(model.rsquared, 3))
print('\n偏回帰係数:'); print(model.params.round(2))
print('\n各係数のp値:'); print(model.pvalues.round(4))

R²≈0.42。`C(Pclass)` の係数から、2等・3等は1等より運賃が大きく下がる（基準=1等）と読めます。

## まとめ

- 3級の「差がありそう」→ **2級の検定で「偶然ではない」と確認**できた（性別・等級と生存、運賃の差すべて有意）。
- 区間推定で「生存率は約35〜42%」のように**幅**で表現できた。
- ANOVA・重回帰で複数グループ・複数要因も扱えた。

> **発展メモ**：`Survived` は0/1なので、「生存の確率」を予測するなら本当は
> **ロジスティック回帰**が適切です（直線回帰だと確率が0〜1をはみ出す）。
> これは2級の少し先（準1級〜）のテーマです。

---
## 練習問題（2級）

**問1.** `Pclass` と `Survived` のカイ二乗 独立性の検定を行い、等級と生存に関連があるか確かめよう。

**問2.** 生存者と死亡者で `Age` の平均に差があるか、Welchの2標本t検定で調べよう（欠損は `dropna()`）。

**問3.** 全体の平均運賃の **99%** 信頼区間を求め、95%のときと幅を比べよう。

**問4.** 重回帰 `Fare ~ C(Pclass) + 家族人数 + C(Embarked)` を作り、決定係数を確認しよう。

In [ ]:
# 問1


In [ ]:
# 問2


In [ ]:
# 問3


In [ ]:
# 問4


> **解答例は別ページにまとめています**（ネタバレ防止）。
> 自分で解いてから **[08_実践_タイタニック分析 の解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/04_%E7%B5%B1%E8%A8%88_2%E7%B4%9A/08_%E5%AE%9F%E8%B7%B5_%E3%82%BF%E3%82%A4%E3%82%BF%E3%83%8B%E3%83%83%E3%82%AF%E5%88%86%E6%9E%90.md)**

## 完了ログ

このノートを終えたら、下のセルに **名前** を入れて実行してください（学習の記録用）。
記録用URL（`LOG_URL`）は配布時に設定されます。空欄のままなら記録されず、メッセージが出るだけです。

In [ ]:
# === 完了ログ ===  このノートを終えたら、NAME を入れて実行してください。
import datetime
try:
    import requests
except ImportError:
    requests = None

NAME = ""       # ← あなたの名前（例: 山田）
LOG_URL = ""    # ← 記録用URL（配布時に設定）
LOG_TOKEN = ""  # ← 記録用トークン（配布時に設定）
NOTEBOOK = "04_統計_2級/08_実践_タイタニック分析"

if NAME and LOG_URL and requests:
    try:
        requests.post(LOG_URL, json={"token": LOG_TOKEN, "name": NAME, "notebook": NOTEBOOK,
                      "time": datetime.datetime.now().isoformat(timespec="seconds")}, timeout=10)
        print(f"記録しました: {NAME} / {NOTEBOOK}")
    except Exception as e:
        print("記録に失敗しました（URL/ネットワークを確認）:", e)
else:
    print(f"{NOTEBOOK}: NAME と LOG_URL を設定すると、学習の完了が記録されます。")